In [1]:
import numpy as np
from typing import Optional
import pickle
from itertools import product

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter, ParameterVector
from qiskit.converters import dag_to_circuit, circuit_to_dag

from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler

from qopt_best_practices.transpilation.qaoa_construction_pass import QAOAConstructionPass

from hubo_qaoa.utils.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from hubo_qaoa.utils.gfa_utils import gfa_file_to_graph
from hubo_qaoa.utils.parameterise_circuit import parameterise_circuit
from hubo_qaoa.utils.str_utils import genbin

from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples

In [68]:
filename: str = 'test_N2_W2'
copy_numbers = [1,1]

rng = np.random.default_rng()

data_file = '/lustre/scratch127/qpg/jc59/new_hubo_formulation/circuit_depths/results.couplingall.precompute.0.pkl'
with open(data_file, 'rb') as f:
    res = pickle.load(f)
cost_circuit = parameterise_circuit(res[filename]['rzz']['circuit'], parameter=Parameter('γ'))
num_qubits: int = cost_circuit.num_qubits    
    
filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)
full_hamiltonian, norm = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)


keys = list(genbin(num_qubits))
evals = evaluate_sparse_pauli_samples(keys, full_hamiltonian) * norm

Keeping constraints at times: [0]


In [69]:
backend_options = dict(
    # method='matrix_product_state',
    # matrix_product_state_max_bond_dimension='20', 
    method='statevector',
    device='CPU',
    precision='single',
    basis_gates = ['rx', 'ry', 'rz', 'cx']
)
# fake_fez = FakeFez()
backend = AerSimulator(**backend_options)
sampler = Sampler.from_backend(backend)



def get_energy(qc):
    job = backend.run([qc],shots=1)
    sampler_result = job.result()
    data = sampler_result.results[0].data

    sv = np.asarray(data.statevector)
    energy = np.sum(np.abs(sv) ** 2 * evals)
    return energy


def LR_QAOA(p: int, delta_b: float, delta_g: float, circ: Optional[QuantumCircuit]):
    betas = [(1-k/p) * delta_b for k in range(p)]
    gammas = [(k+1) / p * delta_g for k in range(p)]
    fixed_params = betas + gammas
    
    if circ is None:
        # construction_pass = QAOAConstructionPass(p, init_state=None, mixer_layer=None)
        # circuit = dag_to_circuit(construction_pass.run(circuit_to_dag(cost_circuit)))
        # circuit.remove_final_measurements()
        # circuit.save_statevector()
        circuit = QuantumCircuit(num_qubits, num_qubits)
        circuit.h(range(num_qubits))
        
        mixer_layer = QuantumCircuit(num_qubits)
        beta = Parameter("β")
        mixer_layer.rx(2 * beta, range(num_qubits))
        
        gamma_params = ParameterVector("γ", p)
        beta_params = ParameterVector("β", p)
        
        for layer in range(p):
            bind_dict = {cost_circuit.parameters[0]: gamma_params[layer]}
            bound_cost_layer = cost_circuit.assign_parameters(bind_dict)

            bind_dict = {mixer_layer.parameters[0]: beta_params[layer]}
            bound_mixer_layer = mixer_layer.assign_parameters(bind_dict)
            
            circuit.compose(bound_cost_layer, range(num_qubits), inplace=True)
            circuit.compose(bound_mixer_layer, range(num_qubits), inplace=True)
            
        circuit.save_statevector()
    else:
        circuit = circ
        
    fixed_param_bind = {circuit.parameters[i]: fixed_params[i] for i in range(2*p)}
    fixed_qc = circuit.assign_parameters(fixed_param_bind)

    energy = get_energy(fixed_qc)
        
    return energy, circuit

In [71]:
e, circ = LR_QAOA(1, 1.56, 1.56, None)
print(e)

9.75277588368981


In [88]:
p = 1
delta_b = 1.
delta_g = 1.
betas = [(1-k/p) * delta_b for k in range(p)]
gammas = [(k+1) / p * delta_g for k in range(p)]
fixed_params = betas + gammas
fixed_param_bind = {circ.parameters[i]: fixed_params[i] for i in range(2*p)}
fixed_qc = circ.assign_parameters(fixed_param_bind)

In [89]:
job = backend.run([fixed_qc],shots=1)
sampler_result = job.result()
data = sampler_result.results[0].data

sv = np.asarray(data.statevector)
energy = np.sum(np.abs(sv) ** 2 * evals)

In [90]:
fixed_qc.draw(fold=-1)

┌───┐                                                                                                  ┌───────┐ statevector 
q_0: ┤ H ├──■───────────────────■──────────■──────────────────────────────────────■────────────────■────────┤ Rx(2) ├──────░──────
     ├───┤┌─┴─┐┌─────────┐┌───┐ │ZZ(-2.5)  │                             ┌───┐  ┌─┴─┐              │        ├───────┤      ░      
q_1: ┤ H ├┤ X ├┤ Rz(2.5) ├┤ X ├─■──────────┼─────────■───────────────────┤ X ├──┤ X ├───■──────────┼────────┤ Rx(2) ├──────░──────
     ├───┤└───┘└─────────┘└─┬─┘            │ZZ(4.5)  │                   └─┬─┘┌─┴───┴─┐ │          │        └───────┘      ░      
q_2: ┤ H ├──────────────────■──────────────■─────────┼─────────■───────────■──┤ Rx(2) ├─┼──────────┼───────────────────────░──────
     ├───┤                                           │ZZ(2.5)  │ZZ(-2.5)      └───────┘ │ZZ(-2.5)  │ZZ(2.5) ┌───────┐      ░      
q_3: ┤ H ├───────────────────────────────────────────■─────────■────────────────────────■──────────■────────┤ Rx(2) ├──────░──────
     └───┘                                                                                                  └───────┘      ░      
c: 4/═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

In [91]:
full_hamiltonian * norm

SparsePauliOp(['IIII', 'IIZZ', 'IZIZ', 'IZZI', 'ZIIZ', 'ZIZI', 'ZZII', 'ZZZZ'],
              coeffs=[ 9.75+0.j,  1.25+0.j,  2.25+0.j, -1.25+0.j,  1.25+0.j, -1.25+0.j,
 -1.25+0.j,  1.25+0.j])

In [92]:
evaluate_sparse_pauli_samples(['0001', '0000'], full_hamiltonian * norm)

array([ 0., 12.])

In [72]:
qc = QuantumCircuit(3)
qc.h(0)
qc.measure_all()
res = backend.run(qc).result()

In [74]:
res.get_counts()

{'000': 537, '001': 487}

In [93]:
def C(x):
    return x[0]*x[1]

In [94]:
from qiskit.quantum_info import SparsePauliOp
H = SparsePauliOp(['III', 'IIZ', 'IZI', 'IZZ'], [0.25, -0.25, -0.25, 0.25])

In [104]:
from qiskit.circuit.library import CXGate, PauliEvolutionGate

qc = QuantumCircuit(3)
qc.h(range(3))
qc.append(PauliEvolutionGate(H), range(3))     
qc.save_statevector()
qc.decompose('*exp*').draw(fold=-1)

global phase: 6.0332
     ┌───┐┌──────────┐           statevector 
q_0: ┤ H ├┤ Rz(-0.5) ├─■──────────────░──────
     ├───┤├──────────┤ │ZZ(0.5)       ░      
q_1: ┤ H ├┤ Rz(-0.5) ├─■──────────────░──────
     ├───┤└──────────┘                ░      
q_2: ┤ H ├────────────────────────────░──────
     └───┘                            ░

In [105]:
res = backend.run(qc.decompose('*exp*')).result()

In [117]:
sv = np.asarray(res.results[0].data.statevector)

In [118]:
for i in range(8):
    binrep = np.binary_repr(i, 3)
    print(binrep)
    print(np.round(np.real(sv[i]), 2), np.round(np.imag(sv[i]), 2))
    print(evaluate_sparse_pauli_samples([binrep], H))

000
0.35 0.0
[0.]
001
0.35 -0.0
[0.]
010
0.35 0.0
[0.]
011
0.19 -0.3
[1.]
100
0.35 0.0
[0.]
101
0.35 -0.0
[0.]
110
0.35 0.0
[0.]
111
0.19 -0.3
[1.]


In [119]:
sv[np.nonzero(evaluate_sparse_pauli_samples([np.binary_repr(i, 3) for i in range(8)], H) == 0)]

array([0.35355335+0.00000000e+00j, 0.35355335-1.58050675e-08j,
       0.35355335+0.00000000e+00j, 0.35355335+0.00000000e+00j,
       0.35355335-1.58050675e-08j, 0.35355335+0.00000000e+00j])

In [4]:
sim = AerSimulator(method='unitary')
qc = QuantumCircuit(1)
qc.ry(-np.pi/2, 0)
qc.rz(-2*np.pi/4, 0)
qc.ry(np.pi/2, 0)
qc.save_unitary()
res = sim.run(qc).result()
res.get_unitary()

Operator([[0.70710678+1.11022302e-16j, 0.        +7.07106781e-01j],
          [0.        +7.07106781e-01j, 0.70710678-1.11022302e-16j]],
         input_dims=(2,), output_dims=(2,))


In [5]:
qc = QuantumCircuit(1)
qc.rx(-2*np.pi/4, 0)
qc.save_unitary()
res = sim.run(qc).result()
res.get_unitary()

Operator([[0.70710678+0.j        , 0.        +0.70710678j],
          [0.        +0.70710678j, 0.70710678+0.j        ]],
         input_dims=(2,), output_dims=(2,))


In [6]:
sim_s = AerSimulator(method='statevector')
qc = QuantumCircuit(1)
qc.ry(np.pi/2, 0)
qc.ry(-np.pi/2, 0)
qc.rz(-2*np.pi/4, 0)
qc.ry(np.pi/2, 0)
qc.save_statevector()
res = sim_s.run(qc).result()
res.get_statevector()

Statevector([0.5+0.5j, 0.5+0.5j],
            dims=(2,))


In [7]:
qc = QuantumCircuit(1)
qc.ry(np.pi/2, 0)
qc.save_statevector()
res = sim_s.run(qc).result()
res.get_statevector()

Statevector([0.70710678+0.j, 0.70710678+0.j],
            dims=(2,))
